# Chrome/ChromeDriver 문제 진단
에러가 발생할 때 이 노트북을 실행하여 정확한 문제를 파악하세요.

## 1단계: 진단 실행

In [ ]:
%%bash
echo "==========================================="
echo "1. Chrome/Chromium 확인"
echo "==========================================="

if command -v google-chrome &> /dev/null; then
    echo "✅ Google Chrome 발견"
    google-chrome --version
elif command -v chromium-browser &> /dev/null; then
    echo "✅ Chromium 발견"
    chromium-browser --version
elif command -v chromium &> /dev/null; then
    echo "✅ Chromium 발견"
    chromium --version
else
    echo "❌ Chrome/Chromium이 설치되지 않았습니다!"
fi

echo ""
echo "==========================================="
echo "2. ChromeDriver 확인"
echo "==========================================="

if command -v chromedriver &> /dev/null; then
    echo "✅ ChromeDriver 발견"
    chromedriver --version 2>&1 | head -1
else
    echo "❌ ChromeDriver가 설치되지 않았습니다!"
fi

echo ""
echo "==========================================="
echo "3. Chrome 바이너리 위치"
echo "==========================================="
which google-chrome chromium-browser chromium 2>/dev/null || echo "없음"

echo ""
echo "==========================================="
echo "4. 실행 중인 Chrome 프로세스"
echo "==========================================="
ps aux | grep -i chrome | grep -v grep || echo "실행 중인 Chrome 프로세스 없음"

## 2단계: Python 패키지 확인

In [ ]:
import sys

print(f"Python 버전: {sys.version}\n")

packages = {
    'selenium': 'Selenium',
    'bs4': 'BeautifulSoup4',
    'webdriver_manager': 'webdriver-manager'
}

print("패키지 설치 상태:")
print("="*40)

for module, name in packages.items():
    try:
        mod = __import__(module)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✅ {name}: {version}")
    except ImportError:
        print(f"❌ {name}: 설치되지 않음")

## 3단계: Chrome 직접 실행 테스트

In [ ]:
%%bash
echo "Chrome headless 모드 테스트..."
echo "==========================================="

# Chrome 바이너리 찾기
CHROME_BIN=""
if [ -f "/usr/bin/google-chrome" ]; then
    CHROME_BIN="/usr/bin/google-chrome"
elif [ -f "/usr/bin/chromium-browser" ]; then
    CHROME_BIN="/usr/bin/chromium-browser"
elif [ -f "/usr/bin/chromium" ]; then
    CHROME_BIN="/usr/bin/chromium"
fi

if [ -z "$CHROME_BIN" ]; then
    echo "❌ Chrome 바이너리를 찾을 수 없습니다!"
    exit 1
fi

echo "사용할 Chrome: $CHROME_BIN"
echo ""

# Chrome 실행 테스트
$CHROME_BIN --headless --no-sandbox --disable-gpu --dump-dom about:blank 2>&1 | head -10

if [ $? -eq 0 ]; then
    echo ""
    echo "✅ Chrome이 정상적으로 실행됩니다!"
else
    echo ""
    echo "❌ Chrome 실행 실패!"
fi

## 4단계: Selenium 테스트

In [ ]:
print("Selenium으로 Chrome 구동 테스트...")
print("="*80)

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    
    # Chrome 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    
    print("\n1️⃣ webdriver-manager 방식 테스트...")
    try:
        from webdriver_manager.chrome import ChromeDriverManager
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.get("about:blank")
        print(f"✅ 성공! 타이틀: {driver.title}")
        driver.quit()
    except Exception as e:
        print(f"❌ 실패: {str(e)[:200]}")
    
    print("\n2️⃣ 기본 방식 테스트...")
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get("about:blank")
        print(f"✅ 성공! 타이틀: {driver.title}")
        driver.quit()
    except Exception as e:
        print(f"❌ 실패: {str(e)[:200]}")
        
except ImportError as e:
    print(f"❌ 패키지 임포트 실패: {e}")

## 해결 방법

### 방법 1: apt-get으로 설치 (가장 안정적)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq chromium-browser chromium-chromedriver
!pip install -q selenium beautifulsoup4

print("\n✅ 설치 완료! 런타임을 재시작하세요.")
print("Runtime > Restart runtime")

### 방법 2: webdriver-manager 사용

In [ ]:
!pip install -q selenium beautifulsoup4 webdriver-manager

print("\n✅ 설치 완료! 런타임을 재시작하세요.")
print("Runtime > Restart runtime")

### 방법 3: 수동으로 Chrome 바이너리 지정

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

chrome_options = Options()
chrome_options.binary_location = "/usr/bin/chromium-browser"  # Chrome 위치 직접 지정
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# ChromeDriver 위치도 직접 지정
service = Service('/usr/lib/chromium-browser/chromedriver')

driver = webdriver.Chrome(service=service, options=chrome_options)
print("✅ 성공!")
driver.quit()